# 19 · Conditional Stability Across Height

This notebook combines two earlier directions:

- **conditional local structure** (after unusually small vs unusually large gaps)
- **height stability** across contiguous zeta-zero blocks

## Core question

If local zeta structure shows a "response" after small vs large gaps, does that effect remain stable as we move higher up the zeros?

## What we track

For each height block and each ensemble (zeta, Poisson, GUE), we measure:

1. conditional 5-gap mean profiles
2. conditional difference profile:
   \[
   \text{after large} - \text{after small}
   \]
3. conditional unevenness shift
4. conditional entropy shift
5. conditional symmetry shift

The main qualitative question is whether zeta and GUE continue to behave similarly across height, while Poisson behaves differently.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window and descriptor helpers

In [ ]:
def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def quantile_thresholds(gaps, q_low=0.2, q_high=0.8):
    return np.quantile(gaps, q_low), np.quantile(gaps, q_high)

def conditional_windows(gaps, low_thr, high_thr, k=5):
    small = []
    large = []
    for i in range(len(gaps) - k + 1):
        w = gaps[i:i+k]
        w_norm = normalize_window(w)
        if gaps[i] <= low_thr:
            small.append(w_norm)
        if gaps[i] >= high_thr:
            large.append(w_norm)
    return np.array(small), np.array(large)

def mean_profile(windows):
    return windows.mean(axis=0)

def descriptor_shift(small_windows, large_windows):
    return {
        "unevenness_shift": float(np.mean([unevenness(w) for w in large_windows]) - np.mean([unevenness(w) for w in small_windows])),
        "entropy_shift": float(np.mean([window_entropy(w) for w in large_windows]) - np.mean([window_entropy(w) for w in small_windows])),
        "symmetry_shift": float(np.mean([reversal_symmetry_score(w) for w in large_windows]) - np.mean([reversal_symmetry_score(w) for w in small_windows])),
    }

## Height blocks

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]
block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]
block_labels

## Compute conditional structure across height blocks

In [ ]:
results = []

for block_start in block_starts:
    zeta_block = zeta_gaps_full[block_start:block_start + block_size]
    poisson_block = poisson_gaps_full[block_start:block_start + block_size]
    gue_block = gue_gaps_full[:max(block_size + 20, 500)]

    for name, gaps in [
        ("zeta", zeta_block),
        ("Poisson", poisson_block),
        ("GUE", gue_block),
    ]:
        g_norm = gaps / gaps.mean()
        low_thr, high_thr = quantile_thresholds(g_norm, q_low=0.2, q_high=0.8)
        small_w, large_w = conditional_windows(g_norm, low_thr, high_thr, k=5)

        prof_small = mean_profile(small_w)
        prof_large = mean_profile(large_w)
        diff_prof = prof_large - prof_small

        shifts = descriptor_shift(small_w, large_w)

        results.append({
            "block_start": block_start,
            "block_label": f"{block_start+1}-{block_start+block_size}",
            "ensemble": name,
            "small_windows": small_w,
            "large_windows": large_w,
            "profile_small": prof_small,
            "profile_large": prof_large,
            "profile_diff": diff_prof,
            **shifts,
        })

len(results)

## Conditional 5-gap difference profile across height

This is the main plot:
\[
\text{after large} - \text{after small}
\]
for each ensemble and each height block.

In [ ]:
fig, axes = plt.subplots(1, len(block_labels), figsize=(18, 4.2), sharey=True)
x = np.arange(1, 6)

for ax, block_label in zip(axes, block_labels):
    rows = [r for r in results if r["block_label"] == block_label]
    for row in rows:
        ax.plot(x, row["profile_diff"], marker="o", label=row["ensemble"])
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_title(block_label)
    ax.set_xlabel("position in 5-gap window")
    ax.set_ylabel("large-minus-small")

axes[0].legend()
plt.tight_layout()
plt.show()

## Descriptor shifts across height

We now track three summary shifts:
- unevenness shift
- entropy shift
- symmetry shift

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), sharex=True)

metrics = ["unevenness_shift", "entropy_shift", "symmetry_shift"]
titles = ["Unevenness shift", "Entropy shift", "Symmetry shift"]

for ax, metric, title in zip(axes, metrics, titles):
    for ensemble in ["zeta", "Poisson", "GUE"]:
        ys = [r[metric] for r in results if r["ensemble"] == ensemble]
        ax.plot(block_labels, ys, marker="o", label=ensemble)
    ax.axhline(0.0, linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_ylabel(metric)

axes[0].legend()
plt.tight_layout()
plt.show()

## Compare zeta and GUE conditional-profile similarity across height

A small profile distance means zeta and GUE retain similar conditional response shapes.

In [ ]:
profile_similarity = []

for block_label in block_labels:
    zeta_row = next(r for r in results if r["block_label"] == block_label and r["ensemble"] == "zeta")
    gue_row = next(r for r in results if r["block_label"] == block_label and r["ensemble"] == "GUE")
    pois_row = next(r for r in results if r["block_label"] == block_label and r["ensemble"] == "Poisson")

    zg = float(np.linalg.norm(zeta_row["profile_diff"] - gue_row["profile_diff"]))
    zp = float(np.linalg.norm(zeta_row["profile_diff"] - pois_row["profile_diff"]))

    profile_similarity.append({
        "block_label": block_label,
        "zeta_GUE_profile_distance": zg,
        "zeta_Poisson_profile_distance": zp,
        "distance_gap": zp - zg,
    })

profile_similarity

In [ ]:
plt.figure(figsize=(8.8, 4.8))
zg = [r["zeta_GUE_profile_distance"] for r in profile_similarity]
zp = [r["zeta_Poisson_profile_distance"] for r in profile_similarity]

plt.plot(block_labels, zg, marker="o", label="zeta–GUE")
plt.plot(block_labels, zp, marker="o", label="zeta–Poisson")
plt.ylabel("conditional-profile distance")
plt.title("Conditional-profile similarity across height")
plt.legend()
plt.show()

## Representative conditional profiles by ensemble

Plot after-small and after-large profiles at two separated height blocks for direct comparison.

In [ ]:
target_blocks = [block_labels[0], block_labels[-1]]
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
x = np.arange(1, 6)

for i, block_label in enumerate(target_blocks):
    for j, ensemble in enumerate(["zeta", "Poisson", "GUE"]):
        row = next(r for r in results if r["block_label"] == block_label and r["ensemble"] == ensemble)
        ax = axes[i, j]
        ax.plot(x, row["profile_small"], marker="o", label="after small")
        ax.plot(x, row["profile_large"], marker="o", label="after large")
        ax.axhline(1.0, linestyle="--", linewidth=1)
        ax.set_title(f"{ensemble} · {block_label}")
        ax.set_xlabel("position")
        ax.set_ylabel("mean normalized gap")

axes[0, 0].legend()
plt.tight_layout()
plt.show()

## Compact summary table

In [ ]:
summary = {
    "profile_similarity": profile_similarity,
    "descriptor_shifts": [
        {
            "block_label": r["block_label"],
            "ensemble": r["ensemble"],
            "unevenness_shift": r["unevenness_shift"],
            "entropy_shift": r["entropy_shift"],
            "symmetry_shift": r["symmetry_shift"],
        }
        for r in results
    ],
}
summary

## Notes

- If zeta and GUE stay closer to each other than zeta and Poisson in conditional-profile distance across height blocks, that supports the idea that the conditional local-response structure is stable, not just the unconditional geometry.
- The descriptor shifts are crude summaries, but they help show whether the conditional effect is systematically different across ensembles.
- This notebook uses contiguous height blocks and quantile thresholds fixed within each block.

## Next directions

A natural next notebook could do one of these:

1. add bootstrap uncertainty bars to the conditional-profile distances  
2. test cross-block transfer for conditional classifiers  
3. compare conditional stability for 4-gap vs 5-gap windows  
4. test whether conditional signals strengthen or weaken at larger heights